# A GAUSSIAN PROCESS REGRESSION FOR FORECASTING SOLID WASTE GENERATION IN UYO MUNICIPALITY

**Candidate:** EFFIONG, SAMUEL ETIM  
**Registration Number:** AK22/PHS/CSC/029  
**Supervisor:** Dr. Anietie P. Ekong  
**Department:** Computer Science, Faculty of Computing and Information Technology  
**Institution:** Akwa Ibom State University, Ikot Akpaden, Nigeria  

---
### Implementation Following Proposed System Architecture (Chapter 3, Figures 3.2 - 3.4)
- **Stage 1: Data Collection & Preprocessing** (AKSEPWMA empirical records 2019-2025, cleaning, imputation, IQR outlier verification, time-series formatting)
- **Stage 2: Model Design & Kernel Optimization** (Temporal cyclical features, weather regressors, demographic index, GPR kernel selection, L-BFGS-B optimization, evaluation metrics)
- **Stage 3: Implementation, Diagnostics & Forecasting** (Actual vs Predicted, Confusion Matrix, Residuals, 2026-2030 projections with 95% Bayesian Confidence Intervals, Streamlit Web App)

## Cell 1: Environment Setup & Library Imports (Step 10 of Methodology)
Import core mathematical, data manipulation, visualization, and Bayesian machine learning libraries.

In [ ]:
# Cell 1: Environment Setup & Library Imports
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, RationalQuadratic, DotProduct, ConstantKernel as C, WhiteKernel
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, confusion_matrix, classification_report
)

# Matplotlib and Seaborn Styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 10

print("[OK] All required libraries imported successfully.")

## Cell 2: Data Collection & Loading (Step 1 & Step 2.1 - 2.3)
Load historical solid waste records obtained from Akwa Ibom State Environmental Protection and Waste Management Agency (AKSEPWMA).

In [ ]:
# Cell 2: Load Data
DATA_PATH = "uyo_msw_data.csv"
df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape[0]} monthly observations, {df.shape[1]} attributes")
print("\n--- First 5 Records ---")
display(df.head())

print("\n--- Missing Values & Data Types ---")
display(df.info())

## Cell 3: Data Preprocessing & Outlier Verification (Step 2.4 - 2.6)
Sort chronological timestamps, verify duplicates, inspect missing public holidays, and perform Interquartile Range (IQR) outlier testing.

In [ ]:
# Cell 3: Preprocessing
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Duplicates verification
duplicates = df.duplicated().sum()
print(f"Duplicate Rows Found: {duplicates}")

# IQR Outlier Detection on Waste Generation
Q1 = df['waste_tonnes_estimated'].quantile(0.25)
Q3 = df['waste_tonnes_estimated'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(df['waste_tonnes_estimated'] < lower_bound) | (df['waste_tonnes_estimated'] > upper_bound)]

print(f"IQR Thresholds: [{lower_bound:.2f}, {upper_bound:.2f}] tonnes")
print(f"Number of Statistical Outliers: {len(outliers)}")

## Cell 4: Exploratory Data Analysis & Trend Visualization (Objective 1)
Visualize historical waste generation trends in Uyo Municipality, calculate per-capita daily rates, and examine correlation structures.

In [ ]:
# Cell 4: Exploratory Analysis
# Compute per-capita generation rate (kg/capita/day)
days_in_mo = df['date'].dt.days_in_month
df['per_capita_kg_day'] = (df['waste_tonnes_estimated'] * 1000) / (df['population'] * days_in_mo)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Historical Waste Tonnage Trend
ax1.plot(df['date'], df['waste_tonnes_estimated'], color='#1E3A8A', lw=2.2, marker='o', markersize=4)
ax1.set_title("AKSEPWMA Monthly Waste Generation (2019 - 2025)", fontweight='bold')
ax1.set_ylabel("Solid Waste (Tonnes)")
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.grid(True, linestyle='--', alpha=0.6)

# Plot 2: Per-Capita Waste Rate
ax2.plot(df['date'], df['per_capita_kg_day'], color='#0D9488', lw=2, marker='s', markersize=4)
ax2.axhline(df['per_capita_kg_day'].mean(), color='red', linestyle='--', label=f"Mean: {df['per_capita_kg_day'].mean():.2f} kg/capita/day")
ax2.set_title("Per Capita Daily Waste Generation Rate in Uyo", fontweight='bold')
ax2.set_ylabel("Rate (kg/person/day)")
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.show()

## Cell 5: Feature Engineering (Step 3)
Construct temporal, cyclical seasonal, climatic, demographic, and calendar regressors.

In [ ]:
# Cell 5: Feature Engineering
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter
df['time_index'] = np.arange(len(df))

# Cyclical month encoding (sin/cos to preserve periodic continuity)
df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12.0)
df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12.0)

# Public holiday count
df['num_holidays'] = df['public_holidays'].fillna('').apply(
    lambda x: len(str(x).split(';')) if str(x).strip() != '' else 0
)

feature_cols = [
    'time_index', 'year', 'month', 'sin_month', 'cos_month',
    'avg_temperature_c', 'rainfall_mm', 'humidity_percent',
    'population', 'num_holidays'
]
target_col = 'waste_tonnes_estimated'

X = df[feature_cols].copy()
y = df[target_col].values

print(f"[OK] Engineered Feature Matrix Shape: {X.shape}")
print("Engineered Features:", feature_cols)

## Cell 6: Chronological Train-Test Split & Standardization (Step 4 & Step 2.5)
Perform an 80/20 chronological time-series split (Train: 2019-01 to 2024-07; Test: 2024-08 to 2025-12) to test real-world out-of-sample forecasting.

In [ ]:
# Cell 6: Train-Test Split & Scaling
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
dates_train, dates_test = df['date'].iloc[:split_idx], df['date'].iloc[split_idx:]

print(f"Training Set: {len(X_train)} months ({dates_train.iloc[0].strftime('%Y-%m')} to {dates_train.iloc[-1].strftime('%Y-%m')})")
print(f"Testing Set:  {len(X_test)} months ({dates_test.iloc[0].strftime('%Y-%m')} to {dates_test.iloc[-1].strftime('%Y-%m')})")

# Feature Scaling with StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_all_scaled = scaler.transform(X)

print("[OK] Features standardized to zero-mean and unit-variance.")

## Cell 7: Kernel Selection & Hyperparameter Optimization (Steps 5 - 8)
Benchmarking candidate covariance functions: RBF, Matérn 1.5, Matérn 2.5, RationalQuadratic, and Composite Kernel.

In [ ]:
# Cell 7: GPR Kernels
candidate_kernels = {
    "RBF + Noise": (
        C(1.0, (1e-2, 1e4)) * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e3))
        + WhiteKernel(noise_level=1e-2, noise_level_bounds=(1e-5, 1e2))
    ),
    "Matern (nu=1.5) + Noise": (
        C(1.0, (1e-2, 1e4)) * Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e3), nu=1.5)
        + WhiteKernel(noise_level=1e-2, noise_level_bounds=(1e-5, 1e2))
    ),
    "Matern (nu=2.5) + Noise": (
        C(1.0, (1e-2, 1e4)) * Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e3), nu=2.5)
        + WhiteKernel(noise_level=1e-2, noise_level_bounds=(1e-5, 1e2))
    ),
    "RationalQuadratic + Noise": (
        C(1.0, (1e-2, 1e4)) * RationalQuadratic(length_scale=1.0, alpha=1.0)
        + WhiteKernel(noise_level=1e-2, noise_level_bounds=(1e-5, 1e2))
    ),
    "Proposed Composite (Trend + Matern + Noise)": (
        C(1.0, (1e-2, 1e4)) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-2, 1e3))
        + C(1.0, (1e-2, 1e4)) * Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e3), nu=1.5)
        + WhiteKernel(noise_level=1e-2, noise_level_bounds=(1e-5, 1e2))
    )
}

def evaluate_model(y_true, y_pred, y_std):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100.0
    r2 = r2_score(y_true, y_pred)
    ci_cov = np.mean((y_true >= (y_pred - 1.96 * y_std)) & (y_true <= (y_pred + 1.96 * y_std))) * 100.0
    return {"RMSE": rmse, "MAE": mae, "MAPE (%)": mape, "R2": r2, "95% CI Coverage (%)": ci_cov}

benchmark_results = []
fitted_models = {}

for name, kern in candidate_kernels.items():
    gpr = GaussianProcessRegressor(kernel=kern, normalize_y=True, alpha=1e-5, n_restarts_optimizer=10, random_state=42)
    gpr.fit(X_train_scaled, y_train)
    pred, std = gpr.predict(X_test_scaled, return_std=True)
    res = evaluate_model(y_test, pred, std)
    res["Kernel"] = name
    benchmark_results.append(res)
    fitted_models[name] = gpr

eval_df = pd.DataFrame(benchmark_results)[["Kernel", "RMSE", "MAE", "MAPE (%)", "R2", "95% CI Coverage (%)"]]
display(eval_df.style.highlight_min(subset=["RMSE", "MAE", "MAPE (%)"], color="#DCFCE7"))

## Cell 8: Model Evaluation & Performance Graphs (Step 9 & Step 10)
Generate **Actual vs Predicted Scatter Plot**, **Prediction Residuals Distribution**, and **Operational Waste Tier Confusion Matrix**.

In [ ]:
# Cell 8: Performance Graphs & Confusion Matrix
best_name = "Proposed Composite (Trend + Matern + Noise)"
best_model = fitted_models[best_name]

y_all_pred, y_all_std = best_model.predict(X_all_scaled, return_std=True)

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# Graph 1: Actual vs Predicted Scatter Plot
ax1.scatter(y[:split_idx], y_all_pred[:split_idx], color='#10B981', alpha=0.7, edgecolors='k', s=50, label='Train Set (2019-2024)')
ax1.scatter(y[split_idx:], y_all_pred[split_idx:], color='#EF4444', alpha=0.8, edgecolors='k', s=60, marker='s', label='Test Set (2024-2025)')
min_v, max_v = min(y.min(), y_all_pred.min()) - 1000, max(y.max(), y_all_pred.max()) + 1000
ax1.plot([min_v, max_v], [min_v, max_v], color='#1E293B', linestyle='--', lw=2, label='Ideal Perfect Fit (y = x)')
ax1.set_xlim([min_v, max_v])
ax1.set_ylim([min_v, max_v])
ax1.set_xlabel("Actual Waste (Tonnes)")
ax1.set_ylabel("GPR Predicted Waste (Tonnes)")
ax1.set_title(f"Actual vs. Predicted Waste (Overall R2 = {r2_score(y, y_all_pred):.4f})", fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

# Graph 2: Residual Normality Distribution
resids = y - y_all_pred
ax2.hist(resids, bins=15, color='#6366F1', edgecolor='black', alpha=0.75, density=True)
mu_r, sig_r = resids.mean(), resids.std()
xr = np.linspace(resids.min() - 500, resids.max() + 500, 100)
pr = (1 / (sig_r * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((xr - mu_r) / sig_r) ** 2)
ax2.plot(xr, pr, color='#DC2626', lw=2, label=f'Gaussian N({mu_r:.1f}, {sig_r:.1f})')
ax2.axvline(0, color='black', linestyle=':', lw=1.5)
ax2.set_xlabel("Residual Error (Actual - Predicted) [Tonnes]")
ax2.set_ylabel("Density")
ax2.set_title("Residual Distribution & Normality Check", fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()

# Graph 3: Time-Series Tracking with 95% Confidence Bounds
ax3.plot(df['date'], y, label='Recorded Actuals', color='#0F172A', lw=2)
ax3.plot(df['date'], y_all_pred, label='GPR Fitted Mean', color='#2563EB', lw=1.8, linestyle='--')
ax3.fill_between(df['date'], y_all_pred - 1.96 * y_all_std, y_all_pred + 1.96 * y_all_std, color='#93C5FD', alpha=0.4, label='95% Confidence Interval')
ax3.axvline(df['date'].iloc[split_idx], color='#DC2626', linestyle=':', lw=2, label='Test Split')
ax3.set_ylabel("Waste (Tonnes)")
ax3.set_title("Time-Series Actual vs GPR Fit & Uncertainty", fontweight='bold')
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax3.grid(True, linestyle='--', alpha=0.6)
ax3.legend(loc='lower right', fontsize=8)

# Graph 4: Operational Waste Tier Confusion Matrix
tier_bins = [-np.inf, 30000, 37000, 41000, np.inf]
tier_labels = ['Low (<30kt)', 'Moderate (30-37kt)', 'High (37-41kt)', 'Critical (>41kt)']
y_cat_true = pd.cut(y, bins=tier_bins, labels=tier_labels)
y_cat_pred = pd.cut(y_all_pred, bins=tier_bins, labels=tier_labels)
cm = confusion_matrix(y_cat_true, y_cat_pred, labels=tier_labels)
cm_acc = np.trace(cm) / np.sum(cm) * 100.0

cax = ax4.imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(cax, ax=ax4)
ax4.set_xticks(range(len(tier_labels)))
ax4.set_yticks(range(len(tier_labels)))
ax4.set_xticklabels(tier_labels, rotation=30, ha='right')
ax4.set_yticklabels(tier_labels)
ax4.set_xlabel("Predicted Operational Tier", fontweight='bold')
ax4.set_ylabel("Actual Operational Tier", fontweight='bold')
ax4.set_title(f"Operational Tier Confusion Matrix (Accuracy: {cm_acc:.1f}%)", fontweight='bold')

for i in range(len(tier_labels)):
    for j in range(len(tier_labels)):
        col = "white" if cm[i, j] > (cm.max() / 2) else "black"
        ax4.text(j, i, format(cm[i, j], 'd'), ha="center", va="center", color=col, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n--- Operational Tier Classification Report ---")
print(classification_report(y_cat_true, y_cat_pred, labels=tier_labels, zero_division=0))

# --- Individual Error Metrics Visualizations (RMSE, MAE, MAPE) ---
k_names = ['RBF', 'Matérn 1.5', 'Matérn 2.5', 'RationalQuad', 'Proposed Composite']
k_colors_rmse = ['#94A3B8', '#94A3B8', '#94A3B8', '#94A3B8', '#059669']
k_colors_mae  = ['#94A3B8', '#94A3B8', '#94A3B8', '#94A3B8', '#2563EB']
k_colors_mape = ['#94A3B8', '#94A3B8', '#94A3B8', '#94A3B8', '#D97706']

# Individual Graph 1: RMSE
plt.figure(figsize=(9, 4.5))
b1 = plt.bar(k_names, eval_df['RMSE'], color=k_colors_rmse, edgecolor='#0F172A', alpha=0.88, width=0.55)
plt.title('Kernel Benchmark: Root Mean Squared Error (RMSE in Tonnes)', fontweight='bold', fontsize=12, pad=12)
plt.ylabel('RMSE (Tonnes)', fontweight='bold')
plt.xlabel('GPR Covariance Kernels', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for bar in b1:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., h + 25, f'{h:,.1f} t', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.ylim(0, eval_df['RMSE'].max() * 1.15)
plt.tight_layout()
plt.show()

# Individual Graph 2: MAE
plt.figure(figsize=(9, 4.5))
b2 = plt.bar(k_names, eval_df['MAE'], color=k_colors_mae, edgecolor='#0F172A', alpha=0.88, width=0.55)
plt.title('Kernel Benchmark: Mean Absolute Error (MAE in Tonnes)', fontweight='bold', fontsize=12, pad=12)
plt.ylabel('MAE (Tonnes)', fontweight='bold')
plt.xlabel('GPR Covariance Kernels', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for bar in b2:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., h + 20, f'{h:,.1f} t', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.ylim(0, eval_df['MAE'].max() * 1.15)
plt.tight_layout()
plt.show()

# Individual Graph 3: MAPE
plt.figure(figsize=(9, 4.5))
b3 = plt.bar(k_names, eval_df['MAPE (%)'], color=k_colors_mape, edgecolor='#0F172A', alpha=0.88, width=0.55)
plt.title('Kernel Benchmark: Mean Absolute Percentage Error (MAPE %)', fontweight='bold', fontsize=12, pad=12)
plt.ylabel('MAPE (%)', fontweight='bold')
plt.xlabel('GPR Covariance Kernels', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.axhline(5.0, color='#DC2626', linestyle='--', lw=1.6, label='High Precision Tolerance Benchmark (5%)')
for bar in b3:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., h + 0.05, f'{h:.2f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.ylim(0, max(eval_df['MAPE (%)'].max() * 1.25, 5.5))
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Cell 9: Model Retraining on Full Dataset & 2026-2030 Projections (Steps 11 - 14)
Retrain the optimal GPR model on the complete 84-month empirical sequence and project future waste generation to 2030.

In [ ]:
# Cell 9: Future Forecasting
final_gpr = GaussianProcessRegressor(
    kernel=candidate_kernels[best_name],
    normalize_y=True,
    alpha=1e-5,
    n_restarts_optimizer=15,
    random_state=42
)
final_scaler = StandardScaler()
X_full_scaled = final_scaler.fit_transform(X)
final_gpr.fit(X_full_scaled, y)

# Construct 2026-2030 Future Dates
future_dates = pd.date_range(start='2026-01-01', end='2030-12-01', freq='MS')
future_df = pd.DataFrame({'date': future_dates})
future_df['year'] = future_df['date'].dt.year
future_df['month'] = future_df['date'].dt.month
future_df['time_index'] = np.arange(len(df), len(df) + len(future_df))
future_df['sin_month'] = np.sin(2 * np.pi * future_df['month'] / 12.0)
future_df['cos_month'] = np.cos(2 * np.pi * future_df['month'] / 12.0)

# Projected Demographic & Climate Features
last_pop = df['population'].iloc[-1]
monthly_growth = (1 + 0.028) ** (1/12) # 2.8% annual population growth
future_df['population'] = [last_pop * (monthly_growth ** (i + 1)) for i in range(len(future_df))]

monthly_climate = df.groupby('month')[['avg_temperature_c', 'rainfall_mm', 'humidity_percent']].mean()
future_df = future_df.merge(monthly_climate, on='month', how='left')

monthly_hols = df.groupby('month')['num_holidays'].median().round()
future_df = future_df.merge(monthly_hols, on='month', how='left')

# Execute Bayesian Inference
X_fut_scaled = final_scaler.transform(future_df[feature_cols])
y_fut_pred, y_fut_std = final_gpr.predict(X_fut_scaled, return_std=True)

future_df['forecast_tonnes'] = y_fut_pred
future_df['daily_tonnes'] = y_fut_pred / 30.4167
future_df['ci_lower_95'] = y_fut_pred - 1.96 * y_fut_std
future_df['ci_upper_95'] = y_fut_pred + 1.96 * y_fut_std
future_df['uncertainty_std'] = y_fut_std

# Option D: 5-Tier Empirical Waste Classification System
def classify_waste_5tier(val):
    if val < 30000:
        return 'Very Low'
    elif val < 35000:
        return 'Low'
    elif val < 40000:
        return 'Moderate'
    elif val < 45000:
        return 'High'
    else:
        return 'Very High'

future_df['waste_class'] = future_df['forecast_tonnes'].apply(classify_waste_5tier)
future_df['compactor_trucks'] = np.ceil(future_df['daily_tonnes'] / 30.0).astype(int)

# Annual Summary Table
annual_proj = future_df.groupby('year').agg(
    Annual_Total=('forecast_tonnes', 'sum'),
    Monthly_Mean=('forecast_tonnes', 'mean'),
    Daily_Mean=('daily_tonnes', 'mean'),
    Lower_95_CI=('ci_lower_95', 'mean'),
    Upper_95_CI=('ci_upper_95', 'mean'),
    Active_Compactors=('compactor_trucks', 'mean')
).reset_index()

print('--- 5-Year Municipal Waste Projections (2026 - 2030) ---')
display(annual_proj.style.format({
    'Annual_Total': '{:,.1f} t',
    'Monthly_Mean': '{:,.1f} t',
    'Daily_Mean': '{:,.1f} t/d',
    'Lower_95_CI': '{:,.1f} t',
    'Upper_95_CI': '{:,.1f} t',
    'Active_Compactors': '{:.0f} Trucks'
}))

# Sample Option D Monthly Forecasts (First 12 Months)
print('\n--- Option D Presentation: Next 12 Months (2026) Forecasts with Assigned Waste Classes ---')
disp_sample = future_df.iloc[:12][['date', 'forecast_tonnes', 'daily_tonnes', 'waste_class', 'ci_lower_95', 'ci_upper_95', 'compactor_trucks']].copy()
disp_sample['date'] = disp_sample['date'].dt.strftime('%B %Y')
display(disp_sample.rename(columns={
    'date': 'Forecast Period',
    'forecast_tonnes': 'Predicted Waste (Tonnes/Mo)',
    'daily_tonnes': 'Daily Waste (Tonnes/Day)',
    'waste_class': 'Waste Generation Class',
    'ci_lower_95': 'Lower 95% Limit',
    'ci_upper_95': 'Upper 95% Limit',
    'compactor_trucks': '15T Compactor Trucks'
}).style.format({
    'Predicted Waste (Tonnes/Mo)': '{:,.1f}',
    'Daily Waste (Tonnes/Day)': '{:,.1f}',
    'Lower 95% Limit': '{:,.1f}',
    'Upper 95% Limit': '{:,.1f}'
}))

# Visual 1: Main Forecast Trajectory with Horizontal Class Bands
plt.figure(figsize=(13, 5.8))
plt.axhspan(0, 30000, color='#DEF7EC', alpha=0.5, label='Very Low (<30k t)')
plt.axhspan(30000, 35000, color='#E0F2FE', alpha=0.5, label='Low (30k-35k t)')
plt.axhspan(35000, 40000, color='#FEF08A', alpha=0.5, label='Moderate (35k-40k t)')
plt.axhspan(40000, 45000, color='#FFEDD5', alpha=0.5, label='High (40k-45k t)')
plt.axhspan(45000, 55000, color='#FEE2E2', alpha=0.5, label='Very High (≥45k t)')
for th in [30000, 35000, 40000, 45000]:
    plt.axhline(th, color='#94A3B8', linestyle=':', lw=0.9, alpha=0.8)
plt.plot(df['date'], df['waste_tonnes_estimated'], color='#0F172A', lw=2.4, marker='o', markersize=3.5, label='Historical Records (2019-2025)')
plt.plot(future_df['date'], future_df['forecast_tonnes'], color='#2563EB', lw=2.6, linestyle='--', marker='s', markersize=3.5, label='GPR Forecast Projections (2026-2030)')
plt.fill_between(future_df['date'], future_df['ci_lower_95'], future_df['ci_upper_95'], color='#93C5FD', alpha=0.45, label='95% Bayesian Confidence Band')
plt.axvline(pd.to_datetime('2026-01-01'), color='#EF4444', linestyle=':', lw=2, label='Forecast Boundary (Jan 2026)')
plt.title('Uyo Municipal Solid Waste Trajectory (2019 - 2030) with Operational Class Zones', fontweight='bold', fontsize=12, pad=12)
plt.ylabel('Solid Waste Generated (Tonnes)', fontweight='bold')
plt.xlabel('Observation Timeline', fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper left', frameon=True, fontsize=8.8)
plt.tight_layout()
plt.show()

# Visual 2: Forecast Class Distribution Breakdown
plt.figure(figsize=(8, 4))
dist_counts = future_df['waste_class'].value_counts().reindex(['Very Low', 'Low', 'Moderate', 'High', 'Very High'], fill_value=0)
dist_colors = ['#059669', '#0284C7', '#D97706', '#EA580C', '#DC2626']
b_dist = plt.bar(dist_counts.index, dist_counts.values, color=dist_colors, edgecolor='#0F172A', alpha=0.88, width=0.55)
plt.title('Forecast Class Distribution: 60 Months (2026 - 2030)', fontweight='bold', fontsize=12, pad=12)
plt.ylabel('Number of Months', fontweight='bold')
plt.xlabel('Operational Waste Generation Class', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for b in b_dist:
    h = b.get_height()
    if h > 0:
        pct = (h / len(future_df)) * 100.0
        plt.text(b.get_x() + b.get_width()/2., h + 0.5, f'{h} ({pct:.0f}%)', ha='center', va='bottom', fontweight='bold')
plt.ylim(0, dist_counts.values.max() * 1.25)
plt.tight_layout()
plt.show()

## Cell 10: Model Serialization & Streamlit Artifact Packaging (Step 11 & Step 15)
Save the trained GPR model, standardizer, and evaluated data structures into `gpr_waste_model.pkl` for the Streamlit dashboard.

In [ ]:
# Cell 10: Model Serialization
train_res = evaluate_model(y_train, best_model.predict(X_train_scaled), y_train*0.01)
test_res = evaluate_model(y_test, best_model.predict(X_test_scaled), y_test*0.01)
overall_res = evaluate_model(y, y_all_pred, y_all_std)

artifacts = {
    'model': final_gpr,
    'scaler': final_scaler,
    'feature_cols': feature_cols,
    'eval_df': eval_df,
    'metrics': {
        'train': train_res,
        'test': test_res,
        'overall': overall_res
    },
    'historical_df': df,
    'future_df': future_df
}

joblib.dump(artifacts, 'gpr_waste_model.pkl')
print("[OK] Successfully saved 'gpr_waste_model.pkl'. Ready for Streamlit deployment.")

## Cell 11: Launch the Interactive Streamlit Web Application (Step 15)
Run the terminal command below to launch the Streamlit user interface in your web browser:

```bash
streamlit run app.py
```

In [ ]:
# Cell 11: Instruction to launch Streamlit dashboard
print("To start the interactive Streamlit dashboard on your computer, open a terminal in this directory and execute:")
print("streamlit run app.py")